In [1]:
# Embedding-based CNN Experiments for Binary Classification
# 
# This notebook experiments with:
# 1. EmbeddingCNN: One-hot (5x600) → Linear transforms (128→64→32) → CNN
# 2. PatchingCNN: Non-overlapping patches for motif detection
# 3. Hyperparameter optimization and comparison

import sys
import os
sys.path.append('../../../')

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

# Import our custom models
from embedding_cnn_models import (
    EmbeddingCNN, PatchingCNN, BinaryClassificationDataset,
    create_embedding_cnn, create_patching_cnn, count_parameters
)

print("Imports successful!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


Imports successful!
PyTorch version: 2.8.0
CUDA available: False
Using device: cpu


In [2]:
# Load and prepare data
print("Loading binary classification data...")

data_binary = pd.read_csv('../../../data/processed/ProSeq_binary_classification.csv')
data_binary = data_binary[['binary_classification', 'ProSeq']]

print(f"Original dataset size: {len(data_binary)}")
sequence_lengths = data_binary['ProSeq'].str.len()
print(f"Sequence length range: {sequence_lengths.min()} to {sequence_lengths.max()}")

# Keep only sequences with length >= 600
data_filtered = data_binary[sequence_lengths >= 600].copy()
print(f"Filtered dataset size (length >= 600): {len(data_filtered)}")

# Check class distribution
class_counts = data_filtered['binary_classification'].value_counts()
print(f"\nClass distribution:")
print(class_counts)
print(f"Class balance: {class_counts.min() / class_counts.max():.3f}")

# Split data
train_data, test_data = train_test_split(data_filtered, test_size=0.2, random_state=42, stratify=data_filtered['binary_classification'])
train_data, val_data = train_test_split(train_data, test_size=0.2, random_state=42, stratify=train_data['binary_classification'])

print(f"\nData splits:")
print(f"Train: {len(train_data)} samples")
print(f"Validation: {len(val_data)} samples") 
print(f"Test: {len(test_data)} samples")

# Create datasets and dataloaders
train_dataset = BinaryClassificationDataset(train_data)
val_dataset = BinaryClassificationDataset(val_data)
test_dataset = BinaryClassificationDataset(test_data)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"\nDataLoaders created with batch size: {batch_size}")

# Test data loading
sample_batch = next(iter(train_loader))
print(f"Sample batch shapes: {sample_batch[0].shape}, {sample_batch[1].shape}")
print(f"Input data type: {sample_batch[0].dtype}")
print(f"Target data type: {sample_batch[1].dtype}")


Loading binary classification data...
Original dataset size: 8307
Sequence length range: 472.0 to 600.0
Filtered dataset size (length >= 600): 8302

Class distribution:
binary_classification
0    5217
1    3085
Name: count, dtype: int64
Class balance: 0.591

Data splits:
Train: 5312 samples
Validation: 1329 samples
Test: 1661 samples

DataLoaders created with batch size: 32
Sample batch shapes: torch.Size([32, 600, 5]), torch.Size([32])
Input data type: torch.float32
Target data type: torch.int64


In [3]:
# Create and compare different model architectures
print("="*60)
print("MODEL ARCHITECTURE COMPARISON")
print("="*60)

# Model configurations to test
model_configs = {
    'EmbeddingCNN_1D_Small': {
        'type': 'embedding',
        'params': {
            'embedding_dims': [64, 32],
            'conv_filters': [32, 64],
            'use_2d_conv': False,
            'dropout': 0.3
        }
    },
    'EmbeddingCNN_1D_Medium': {
        'type': 'embedding', 
        'params': {
            'embedding_dims': [128, 64, 32],
            'conv_filters': [64, 128, 256],
            'use_2d_conv': False,
            'dropout': 0.3
        }
    },
    'EmbeddingCNN_2D_Medium': {
        'type': 'embedding',
        'params': {
            'embedding_dims': [128, 64, 32],
            'conv_filters': [64, 128, 256],
            'use_2d_conv': True,
            'dropout': 0.3
        }
    },
    'PatchingCNN_Small': {
        'type': 'patching',
        'params': {
            'patch_size': 3,
            'patch_embed_dim': 16,
            'conv_filters': [32, 64],
            'motif_sizes': [6, 8, 10],
            'dropout': 0.3
        }
    },
    'PatchingCNN_Medium': {
        'type': 'patching',
        'params': {
            'patch_size': 3,
            'patch_embed_dim': 20,
            'conv_filters': [32, 64, 128],
            'motif_sizes': [6, 7, 8, 9, 10, 11],
            'dropout': 0.3
        }
    }
}

# Create models and analyze
models = {}
model_stats = {}

for name, config in model_configs.items():
    print(f"\nCreating {name}...")
    
    if config['type'] == 'embedding':
        model = create_embedding_cnn(**config['params'])
    else:  # patching
        model = create_patching_cnn(**config['params'])
    
    models[name] = model
    param_count = count_parameters(model)
    model_stats[name] = {
        'parameters': param_count,
        'type': config['type'],
        'samples_per_param': len(train_data) / param_count
    }
    
    print(f"  Parameters: {param_count:,}")
    print(f"  Samples per parameter: {len(train_data) / param_count:.1f}")

# Display comparison table
print(f"\n{'Model':<25} {'Type':<10} {'Parameters':<12} {'Samples/Param':<15}")
print("-" * 65)
for name, stats in model_stats.items():
    print(f"{name:<25} {stats['type']:<10} {stats['parameters']:<12,} {stats['samples_per_param']:<15.1f}")

# Test forward pass with sample data
print(f"\nTesting forward pass with sample batch...")
sample_x, sample_y = sample_batch
sample_x = sample_x.to(device)

for name, model in models.items():
    model = model.to(device)
    model.eval()
    with torch.no_grad():
        try:
            output = model(sample_x)
            print(f"{name}: Output shape {output.shape} ✓")
        except Exception as e:
            print(f"{name}: Error - {e} ✗")


MODEL ARCHITECTURE COMPARISON

Creating EmbeddingCNN_1D_Small...
  Parameters: 1,257,409
  Samples per parameter: 0.0

Creating EmbeddingCNN_1D_Medium...
  Parameters: 2,632,097
  Samples per parameter: 0.0

Creating EmbeddingCNN_2D_Medium...
  Parameters: 49,675,745
  Samples per parameter: 0.0

Creating PatchingCNN_Small...
  Parameters: 904,785
  Samples per parameter: 0.0

Creating PatchingCNN_Medium...
  Parameters: 976,277
  Samples per parameter: 0.0

Model                     Type       Parameters   Samples/Param  
-----------------------------------------------------------------
EmbeddingCNN_1D_Small     embedding  1,257,409    0.0            
EmbeddingCNN_1D_Medium    embedding  2,632,097    0.0            
EmbeddingCNN_2D_Medium    embedding  49,675,745   0.0            
PatchingCNN_Small         patching   904,785      0.0            
PatchingCNN_Medium        patching   976,277      0.0            

Testing forward pass with sample batch...
EmbeddingCNN_1D_Small: Output sh

In [4]:
# Training function
def train_model(model, train_loader, val_loader, num_epochs=20, lr=0.001, patience=5):
    """Train a model with early stopping."""
    model = model.to(device)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    
    train_losses = []
    val_losses = []
    val_accuracies = []
    
    best_val_loss = float('inf')
    patience_counter = 0
    best_model_state = None
    
    print(f"Training for up to {num_epochs} epochs...")
    
    for epoch in range(num_epochs):
        # Training phase
        model.train()
        epoch_train_loss = 0.0
        
        for batch_x, batch_y in train_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.float().to(device)
            
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            epoch_train_loss += loss.item()
        
        # Validation phase
        model.eval()
        epoch_val_loss = 0.0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x = batch_x.to(device)
                batch_y = batch_y.float().to(device)
                
                outputs = model(batch_x)
                loss = criterion(outputs, batch_y)
                epoch_val_loss += loss.item()
                
                # Calculate accuracy
                predicted = (outputs > 0.5).float()
                total += batch_y.size(0)
                correct += (predicted == batch_y).sum().item()
        
        # Calculate averages
        avg_train_loss = epoch_train_loss / len(train_loader)
        avg_val_loss = epoch_val_loss / len(val_loader)
        val_accuracy = correct / total
        
        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)
        val_accuracies.append(val_accuracy)
        
        # Early stopping check
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
            best_model_state = model.state_dict().copy()
        else:
            patience_counter += 1
        
        # Print progress
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f'Epoch {epoch+1:2d}/{num_epochs}: Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}, Val Acc: {val_accuracy:.4f}')
        
        # Early stopping
        if patience_counter >= patience:
            print(f'Early stopping at epoch {epoch+1}')
            break
    
    # Load best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    
    return {
        'train_losses': train_losses,
        'val_losses': val_losses,
        'val_accuracies': val_accuracies,
        'best_val_loss': best_val_loss,
        'final_val_accuracy': max(val_accuracies)
    }


def evaluate_model(model, test_loader):
    """Evaluate model on test set."""
    model.eval()
    all_predictions = []
    all_targets = []
    test_loss = 0.0
    criterion = nn.BCELoss()
    
    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.float().to(device)
            
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            test_loss += loss.item()
            
            predictions = (outputs > 0.5).float()
            all_predictions.extend(predictions.cpu().numpy())
            all_targets.extend(batch_y.cpu().numpy())
    
    avg_test_loss = test_loss / len(test_loader)
    accuracy = accuracy_score(all_targets, all_predictions)
    
    return {
        'test_loss': avg_test_loss,
        'accuracy': accuracy,
        'predictions': all_predictions,
        'targets': all_targets,
        'classification_report': classification_report(all_targets, all_predictions)
    }

print("Training and evaluation functions defined!")


Training and evaluation functions defined!


In [ ]:
# Train and compare models
print("="*60)
print("TRAINING MODELS")
print("="*60)

# Select models to train (start with smaller ones for faster experimentation)
models_to_train = ['EmbeddingCNN_1D_Small', 'PatchingCNN_Small', 'EmbeddingCNN_1D_Medium']

training_results = {}

for model_name in models_to_train:
    print(f"\n{'='*20} Training {model_name} {'='*20}")
    
    model = models[model_name]
    
    # Train the model
    results = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        num_epochs=30,
        lr=0.001,
        patience=8
    )
    
    # Evaluate on test set
    test_results = evaluate_model(model, test_loader)
    
    # Combine results
    training_results[model_name] = {
        'training': results,
        'test': test_results,
        'parameters': model_stats[model_name]['parameters']
    }
    
    print(f"\nFinal Results for {model_name}:")
    print(f"  Best Validation Loss: {results['best_val_loss']:.4f}")
    print(f"  Final Validation Accuracy: {results['final_val_accuracy']:.4f}")
    print(f"  Test Accuracy: {test_results['accuracy']:.4f}")
    print(f"  Parameters: {model_stats[model_name]['parameters']:,}")

print(f"\n{'='*60}")
print("TRAINING COMPLETE")
print(f"{'='*60}")


TRAINING MODELS

==================== Training EmbeddingCNN_1D_Small ====================
Training for up to 30 epochs...
Epoch  1/30: Train Loss: 0.6846, Val Loss: 0.6726, Val Acc: 0.6283


In [ ]:
# Visualization and analysis
def plot_training_results(training_results):
    """Plot comprehensive training results comparison."""
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    # Plot 1: Training curves
    ax = axes[0, 0]
    for model_name, results in training_results.items():
        epochs = range(1, len(results['training']['train_losses']) + 1)
        ax.plot(epochs, results['training']['train_losses'], 
                label=f'{model_name} Train', linestyle='-', alpha=0.7)
        ax.plot(epochs, results['training']['val_losses'], 
                label=f'{model_name} Val', linestyle='--', alpha=0.7)
    
    ax.set_title('Training and Validation Loss')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.grid(True, alpha=0.3)
    
    # Plot 2: Validation accuracy curves
    ax = axes[0, 1]
    for model_name, results in training_results.items():
        epochs = range(1, len(results['training']['val_accuracies']) + 1)
        ax.plot(epochs, results['training']['val_accuracies'], 
                label=model_name, marker='o', markersize=3)
    
    ax.set_title('Validation Accuracy')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Plot 3: Test accuracy comparison
    ax = axes[0, 2]
    model_names = list(training_results.keys())
    test_accuracies = [training_results[name]['test']['accuracy'] for name in model_names]
    
    bars = ax.bar(range(len(model_names)), test_accuracies, 
                  color=['lightblue', 'lightgreen', 'lightcoral'][:len(model_names)])
    ax.set_title('Test Accuracy Comparison')
    ax.set_ylabel('Accuracy')
    ax.set_xticks(range(len(model_names)))
    ax.set_xticklabels([name.replace('_', '\n') for name in model_names], rotation=45)
    
    # Add value labels on bars
    for bar, acc in zip(bars, test_accuracies):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
                f'{acc:.3f}', ha='center', va='bottom')
    
    # Plot 4: Parameter efficiency
    ax = axes[1, 0]
    param_counts = [training_results[name]['parameters'] for name in model_names]
    efficiency = [acc * 1000 / params for acc, params in zip(test_accuracies, param_counts)]
    
    bars = ax.bar(range(len(model_names)), efficiency,
                  color=['lightblue', 'lightgreen', 'lightcoral'][:len(model_names)])
    ax.set_title('Parameter Efficiency\n(Accuracy × 1000 / Parameters)')
    ax.set_ylabel('Efficiency Score')
    ax.set_xticks(range(len(model_names)))
    ax.set_xticklabels([name.replace('_', '\n') for name in model_names], rotation=45)
    
    for bar, eff in zip(bars, efficiency):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(efficiency)*0.01, 
                f'{eff:.2f}', ha='center', va='bottom')
    
    # Plot 5: Parameter count comparison (log scale)
    ax = axes[1, 1]
    bars = ax.bar(range(len(model_names)), param_counts,
                  color=['lightblue', 'lightgreen', 'lightcoral'][:len(model_names)])
    ax.set_title('Model Parameter Count')
    ax.set_ylabel('Parameters (log scale)')
    ax.set_yscale('log')
    ax.set_xticks(range(len(model_names)))
    ax.set_xticklabels([name.replace('_', '\n') for name in model_names], rotation=45)
    
    for bar, params in zip(bars, param_counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.1, 
                f'{params:,}', ha='center', va='bottom', rotation=45)
    
    # Plot 6: Overfitting analysis
    ax = axes[1, 2]
    overfitting = []
    for model_name in model_names:
        final_train_loss = training_results[model_name]['training']['train_losses'][-1]
        final_val_loss = training_results[model_name]['training']['val_losses'][-1]
        overfitting.append(final_val_loss - final_train_loss)
    
    bars = ax.bar(range(len(model_names)), overfitting,
                  color=['red' if x > 0.5 else 'orange' if x > 0.1 else 'green' for x in overfitting])
    ax.set_title('Overfitting Assessment\n(Val Loss - Train Loss)')
    ax.set_ylabel('Loss Difference')
    ax.set_xticks(range(len(model_names)))
    ax.set_xticklabels([name.replace('_', '\n') for name in model_names], rotation=45)
    ax.axhline(y=0, color='black', linestyle='-', alpha=0.3)
    
    for bar, over in zip(bars, overfitting):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
                f'{over:.3f}', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()

# Generate plots
plot_training_results(training_results)

# Print detailed comparison
print("\n" + "="*80)
print("DETAILED MODEL COMPARISON")
print("="*80)

for model_name, results in training_results.items():
    print(f"\n{model_name}:")
    print(f"  Architecture: {model_stats[model_name]['type']}")
    print(f"  Parameters: {results['parameters']:,}")
    print(f"  Samples per parameter: {len(train_data) / results['parameters']:.1f}")
    print(f"  Best validation loss: {results['training']['best_val_loss']:.4f}")
    print(f"  Final validation accuracy: {results['training']['final_val_accuracy']:.4f}")
    print(f"  Test accuracy: {results['test']['accuracy']:.4f}")
    print(f"  Test loss: {results['test']['test_loss']:.4f}")
    
    # Overfitting assessment
    final_train_loss = results['training']['train_losses'][-1]
    final_val_loss = results['training']['val_losses'][-1]
    overfitting = final_val_loss - final_train_loss
    print(f"  Overfitting (Val-Train loss): {overfitting:.4f}")
    
    if overfitting < 0.1:
        print("  Good generalization")
    elif overfitting < 0.5:
        print("  Moderate overfitting")
    else:
        print("  Significant overfitting")

# Best model recommendation
best_model = max(training_results.keys(), 
                key=lambda x: training_results[x]['test']['accuracy'])
print(f"\n🏆 BEST MODEL: {best_model}")
print(f"   Test Accuracy: {training_results[best_model]['test']['accuracy']:.4f}")
print(f"   Parameters: {training_results[best_model]['parameters']:,}")

# Show classification reports
print(f"\n{'='*60}")
print("CLASSIFICATION REPORTS")
print(f"{'='*60}")

for model_name, results in training_results.items():
    print(f"\n{model_name}:")
    print(results['test']['classification_report'])
